# MCP Clinical Pipeline Evaluation\nThis notebook runs the WHO Classification, RANO Criteria, and CAP Structured Reporting modules on the radiomics data extracted from the BraTS 2020 dataset.

## 1. Setup the Environment\nWe need to write the MCP modules to the local Colab filesystem so we can import them.

In [ ]:
import os
os.makedirs('mcp_servers', exist_ok=True)
os.makedirs('reports/cap', exist_ok=True)
print('Directories created.')

In [ ]:
%%writefile mcp_servers/who_classification.py
"""
WHO CNS Tumor Classification MCP Server
==========================================
Implements WHO CNS5 (2021) classification with:
  - Molecular markers (IDH, MGMT, 1p/19q, ATRX, TP53)
  - Low-confidence detection via score gap analysis
  - Top-5 differential diagnosis
  - Sigmoid-calibrated confidence scoring
  - Uncertainty estimation

Can run as standalone MCP server or direct function call.
"""
import math
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("WHO_Classification")


def _get_who_config():
    try:
        from config.config_loader import get_config
        return get_config().who_classification
    except Exception:
        from dataclasses import dataclass, field
        from typing import Dict
        @dataclass
        class _D:
            differential_top_n: int = 5
            low_confidence_gap_threshold: float = 1.0
            calibration_method: str = "sigmoid"
            sigmoid_k: float = 1.2
            sigmoid_midpoint: float = 4.0
            default_molecular_markers: Dict[str, str] = field(default_factory=lambda: {
                "IDH": "unknown", "MGMT": "unknown", "1p19q": "unknown",
                "ATRX": "unknown", "TP53": "unknown",
            })
        return _D()


# --- Molecular Marker Profiles ---------------------------------------

MOLECULAR_PROFILES = {
    "glioblastoma": {
        "IDH": "wildtype", "MGMT": "variable", "1p19q": "intact",
        "ATRX": "retained", "TP53": "variable",
    },
    "astrocytoma": {
        "IDH": "mutant", "MGMT": "variable", "1p19q": "intact",
        "ATRX": "lost", "TP53": "mutant",
    },
    "oligodendroglioma": {
        "IDH": "mutant", "MGMT": "methylated", "1p19q": "codeleted",
        "ATRX": "retained", "TP53": "wildtype",
    },
    "meningioma": {
        "IDH": "wildtype", "MGMT": "unmethylated", "1p19q": "intact",
        "ATRX": "retained", "TP53": "wildtype",
    },
    "glioma_nos": {
        "IDH": "unknown", "MGMT": "unknown", "1p19q": "unknown",
        "ATRX": "unknown", "TP53": "unknown",
    },
}


# --- WHO CNS5 Classification Knowledge Base --------------------------

WHO_CLASSIFICATIONS = {
    "glioblastoma": {
        "who_grade": "IV",
        "full_name": "Glioblastoma, IDH-wildtype",
        "description": (
            "Highly malignant diffuse astrocytic glioma. "
            "Characterized by microvascular proliferation and/or necrosis. "
            "Most common primary malignant brain tumor in adults."
        ),
        "typical_features": {
            "volume": "large", "enhancement": "ring-enhancing with central necrosis",
            "growth_rate": "rapid", "sphericity": "irregular (low sphericity)",
            "typical_location": ["frontal", "temporal", "parietal"],
        },
        "morphology_indicators": {
            "min_volume": 15000, "max_sphericity": 0.6,
            "intensity_heterogeneity": "high",
        },
        "molecular_profile": MOLECULAR_PROFILES["glioblastoma"],
        "prognosis": "Poor. Median survival 14-16 months with standard treatment.",
        "standard_treatment": "Maximal safe resection, radiotherapy, temozolomide.",
    },
    "astrocytoma": {
        "who_grade": "II-III",
        "full_name": "Astrocytoma, IDH-mutant",
        "description": (
            "Diffuse astrocytic glioma with IDH mutation. "
            "Grade II (low-grade) or Grade III (anaplastic). "
            "Better prognosis than glioblastoma."
        ),
        "typical_features": {
            "volume": "small to medium",
            "enhancement": "minimal or no enhancement (grade II), variable (grade III)",
            "growth_rate": "slow to moderate", "sphericity": "moderate",
            "typical_location": ["frontal", "temporal"],
        },
        "morphology_indicators": {
            "min_volume": 3000, "max_volume": 40000, "min_sphericity": 0.4,
        },
        "molecular_profile": MOLECULAR_PROFILES["astrocytoma"],
        "prognosis": "Variable. Grade II median survival 7-10 years. Grade III: 3-5 years.",
        "standard_treatment": "Surgery when feasible, radiation, chemotherapy for higher grades.",
    },
    "oligodendroglioma": {
        "who_grade": "II-III",
        "full_name": "Oligodendroglioma, IDH-mutant, 1p/19q-codeleted",
        "description": (
            "Diffuse glioma with IDH mutation and 1p/19q codeletion. "
            "Characteristically demonstrates calcifications on imaging. "
            "Generally better prognosis among diffuse gliomas."
        ),
        "typical_features": {
            "volume": "small to medium",
            "enhancement": "variable, often cortical involvement",
            "growth_rate": "slow", "sphericity": "moderate to high",
            "typical_location": ["frontal"],
        },
        "morphology_indicators": {
            "min_volume": 2000, "max_volume": 35000, "min_sphericity": 0.5,
        },
        "molecular_profile": MOLECULAR_PROFILES["oligodendroglioma"],
        "prognosis": "Favorable. Grade II median survival >10 years.",
        "standard_treatment": "Surgery, PCV chemotherapy, radiation.",
    },
    "meningioma": {
        "who_grade": "I-III",
        "full_name": "Meningioma",
        "description": (
            "Extra-axial tumor arising from meningothelial cells. "
            "Most common primary intracranial tumor. "
            "Majority are WHO Grade I (benign)."
        ),
        "typical_features": {
            "volume": "variable",
            "enhancement": "homogeneous, intense enhancement",
            "growth_rate": "slow",
            "sphericity": "high (well-circumscribed)",
            "typical_location": ["parietal", "frontal"],
        },
        "morphology_indicators": {"min_sphericity": 0.65},
        "molecular_profile": MOLECULAR_PROFILES["meningioma"],
        "prognosis": "Excellent for grade I. Grade II/III have higher recurrence.",
        "standard_treatment": "Surgical resection. Radiation for incompletely resected or higher grade.",
    },
    "glioma_nos": {
        "who_grade": "II-IV",
        "full_name": "Glioma, not otherwise specified",
        "description": (
            "Diffuse glioma that cannot be further classified due to "
            "insufficient molecular data. Grading based on histology."
        ),
        "typical_features": {
            "volume": "variable", "enhancement": "variable",
            "growth_rate": "variable", "sphericity": "variable",
            "typical_location": ["frontal", "temporal", "parietal", "occipital"],
        },
        "morphology_indicators": {},
        "molecular_profile": MOLECULAR_PROFILES["glioma_nos"],
        "prognosis": "Depends on histological grade.",
        "standard_treatment": "Surgery, radiation and/or chemotherapy based on grade.",
    },
}


def _sigmoid_confidence(score: float, k: float = 1.2,
                        midpoint: float = 4.0) -> float:
    """Sigmoid-calibrated confidence: prevents overconfident low scores."""
    try:
        return 1.0 / (1.0 + math.exp(-k * (score - midpoint)))
    except OverflowError:
        return 0.0 if score < midpoint else 1.0


def _compute_molecular_score(tumor_type: str,
                              molecular_markers: dict) -> tuple:
    """Score molecular marker concordance. Returns (score, reasons)."""
    expected = MOLECULAR_PROFILES.get(tumor_type, {})
    score = 0.0
    reasons = []

    for marker, patient_status in molecular_markers.items():
        if patient_status == "unknown":
            continue
        expected_status = expected.get(marker, "unknown")
        if expected_status == "unknown" or expected_status == "variable":
            continue

        if patient_status.lower() == expected_status.lower():
            bonus = 2.0 if marker == "IDH" else 1.5
            score += bonus
            reasons.append(
                f"{marker} {patient_status} matches {tumor_type} profile (+{bonus})"
            )
        else:
            penalty = -1.5 if marker == "IDH" else -0.5
            score += penalty
            reasons.append(
                f"{marker} {patient_status} contradicts {tumor_type} "
                f"(expected {expected_status}, {penalty})"
            )

    return score, reasons


def classify_tumor(morphology: dict, radiomics_patterns: dict = None,
                   growth_characteristics: dict = None,
                   molecular_markers: dict = None) -> dict:
    """
    Classify a tumor using WHO CNS5 guidelines.

    Args:
        morphology: dict with tumor_volume, max_diameter, sphericity, surface_area
        radiomics_patterns: dict with intensity/texture features (optional)
        growth_characteristics: dict with growth_rate, enhancement info (optional)
        molecular_markers: dict with IDH, MGMT, 1p19q, ATRX, TP53 status (optional)

    Returns:
        dict with classification, confidence, reasoning, differential, molecular info
    """
    cfg = _get_who_config()
    radiomics_patterns = radiomics_patterns or {}
    growth_characteristics = growth_characteristics or {}
    molecular_markers = molecular_markers or cfg.default_molecular_markers

    volume = morphology.get("tumor_volume", 0)
    sphericity = morphology.get("sphericity", 0.5)

    scores = {}
    reasoning = {}

    for tumor_type, info in WHO_CLASSIFICATIONS.items():
        score = 0.0
        reasons = []
        indicators = info["morphology_indicators"]

        # Volume scoring
        min_vol = indicators.get("min_volume", 0)
        max_vol = indicators.get("max_volume", float("inf"))
        if min_vol <= volume <= max_vol:
            score += 2.0
            reasons.append(f"Volume {volume:.0f} mm3 matches {tumor_type} range")
        elif volume > 0:
            if volume > min_vol * 0.5:
                score += 0.5

        # Sphericity scoring
        min_sph = indicators.get("min_sphericity", 0)
        max_sph = indicators.get("max_sphericity", 1.0)
        if min_sph <= sphericity <= max_sph:
            score += 2.0
            reasons.append(f"Sphericity {sphericity:.3f} consistent with {tumor_type}")
        elif sphericity > 0:
            score += 0.5

        # Size-specific scoring
        if tumor_type == "glioblastoma" and volume > 30000:
            score += 2.0
            reasons.append("Large volume strongly suggests high-grade lesion")
        elif tumor_type == "meningioma" and sphericity > 0.7:
            score += 2.0
            reasons.append("High sphericity suggests well-circumscribed extra-axial mass")

        # Growth rate
        growth_rate = growth_characteristics.get("growth_rate", None)
        if growth_rate is not None:
            typical_growth = info["typical_features"]["growth_rate"]
            if "rapid" in typical_growth and growth_rate > 0.5:
                score += 1.5
                reasons.append("Rapid growth rate matches profile")
            elif "slow" in typical_growth and growth_rate < 0.1:
                score += 1.5
                reasons.append("Slow growth rate matches profile")

        # Intensity heterogeneity
        intensity_std = radiomics_patterns.get("intensity_std", 0)
        if tumor_type == "glioblastoma" and intensity_std > 0.5:
            score += 1.0
            reasons.append("High intensity heterogeneity consistent with GBM")
        elif tumor_type in ["astrocytoma", "oligodendroglioma"] and intensity_std < 0.5:
            score += 1.0
            reasons.append("Low intensity heterogeneity consistent with lower-grade glioma")

        # Molecular markers scoring
        mol_score, mol_reasons = _compute_molecular_score(tumor_type, molecular_markers)
        score += mol_score
        reasons.extend(mol_reasons)

        scores[tumor_type] = score
        reasoning[tumor_type] = reasons

    # Sort by score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_type = ranked[0][0]
    top_score = ranked[0][1]
    second_score = ranked[1][1] if len(ranked) > 1 else 0

    # Confidence calibration
    if cfg.calibration_method == "sigmoid":
        confidence = _sigmoid_confidence(top_score, cfg.sigmoid_k, cfg.sigmoid_midpoint)
    else:
        max_possible = 12.0  # Updated for molecular markers
        confidence = min(top_score / max_possible, 1.0)

    # Low-confidence detection
    confidence_gap = top_score - second_score
    low_confidence = confidence_gap < cfg.low_confidence_gap_threshold

    # Uncertainty estimation
    all_scores = [s for _, s in ranked if s > 0]
    if len(all_scores) > 1:
        import statistics
        score_spread = statistics.stdev(all_scores)
    else:
        score_spread = 0.0

    # Build result
    classification = WHO_CLASSIFICATIONS[top_type].copy()
    classification["classified_as"] = top_type
    classification["confidence"] = round(confidence, 3)
    classification["confidence_gap"] = round(confidence_gap, 2)
    classification["low_confidence"] = low_confidence
    classification["reasoning"] = reasoning[top_type]

    if low_confidence:
        classification["reasoning"].append(
            f"[WARNING] Low confidence: score gap ({confidence_gap:.1f}) below "
            f"threshold ({cfg.low_confidence_gap_threshold}). "
            "Molecular testing strongly recommended."
        )

    # Top-N differential diagnosis
    diff_n = cfg.differential_top_n
    classification["differential"] = [
        {"type": t, "score": round(s, 2), "reasoning": reasoning[t]}
        for t, s in ranked[1:diff_n + 1]
        if s > 0
    ]

    # Molecular markers used
    classification["molecular_markers_input"] = molecular_markers
    classification["molecular_profile_expected"] = MOLECULAR_PROFILES.get(top_type, {})

    # Uncertainty estimation
    classification["uncertainty"] = {
        "score_spread": round(score_spread, 2),
        "confidence_interval": [
            round(max(0, confidence - 0.1 * score_spread), 3),
            round(min(1, confidence + 0.1 * score_spread), 3),
        ],
    }

    return classification


# --- MCP Server (optional standalone mode) ----------------------------

def run_mcp_server():
    """Run as a standalone MCP server via stdio transport."""
    try:
        from mcp.server.fastmcp import FastMCP
    except ImportError:
        print("ERROR: 'mcp' package not installed. pip install mcp[cli]")
        return

    mcp = FastMCP("WHO Tumor Classification")

    @mcp.tool()
    def who_classify_tumor(
        tumor_volume: float = 0, sphericity: float = 0.5,
        max_diameter: float = 0, surface_area: float = 0,
        intensity_std: float = 0, intensity_skewness: float = 0,
        growth_rate: float = None,
        idh_status: str = "unknown", mgmt_status: str = "unknown",
        codeletion_1p19q: str = "unknown",
        atrx_status: str = "unknown", tp53_status: str = "unknown",
    ) -> dict:
        """Classify a brain tumor using WHO CNS5 guidelines with molecular markers."""
        morphology = {
            "tumor_volume": tumor_volume, "sphericity": sphericity,
            "max_diameter": max_diameter, "surface_area": surface_area,
        }
        radiomics = {"intensity_std": intensity_std, "intensity_skewness": intensity_skewness}
        growth = {"growth_rate": growth_rate} if growth_rate is not None else {}
        molecular = {
            "IDH": idh_status, "MGMT": mgmt_status, "1p19q": codeletion_1p19q,
            "ATRX": atrx_status, "TP53": tp53_status,
        }
        return classify_tumor(morphology, radiomics, growth, molecular)

    mcp.run(transport="stdio")


if __name__ == "__main__":
    run_mcp_server()


In [ ]:
%%writefile mcp_servers/rano_criteria.py
"""
RANO Criteria MCP Server
==========================
Implements RANO (Response Assessment in Neuro-Oncology) criteria
for evaluating tumor treatment response.

Can run as:
  - Standalone MCP server (stdio transport)
  - Direct function call (for in-process use)

RANO Response Categories:
  CR - Complete Response
  PR - Partial Response
  SD - Stable Disease
  PD - Progressive Disease
"""

# --- RANO Criteria Knowledge Base -------------------------------------

RANO_CRITERIA = {
    "CR": {
        "name": "Complete Response",
        "description": (
            "Complete disappearance of all enhancing measurable and "
            "non-measurable disease sustained for at least 4 weeks. "
            "No new lesions. Stable or improved non-enhancing FLAIR/T2 lesions. "
            "Patient off corticosteroids or on physiologic replacement. "
            "Clinically stable or improved."
        ),
        "requirements": {
            "enhancing_tumor": "Complete disappearance",
            "non_enhancing_tumor": "Stable or decreased",
            "new_lesions": "None",
            "corticosteroids": "None or physiologic replacement",
            "clinical_status": "Stable or improved",
        },
    },
    "PR": {
        "name": "Partial Response",
        "description": (
            ">=50% decrease in the sum of products of perpendicular diameters "
            "of all measurable enhancing lesions sustained for at least 4 weeks. "
            "No progression of non-measurable disease. No new lesions. "
            "Stable or reduced corticosteroid dose. Clinically stable or improved."
        ),
        "requirements": {
            "enhancing_tumor": ">=50% decrease",
            "non_enhancing_tumor": "Stable or decreased",
            "new_lesions": "None",
            "corticosteroids": "Stable or decreased",
            "clinical_status": "Stable or improved",
        },
    },
    "SD": {
        "name": "Stable Disease",
        "description": (
            "Does not qualify for complete response, partial response, "
            "or progressive disease. Stable non-enhancing FLAIR/T2 lesions. "
            "Clinically stable."
        ),
        "requirements": {
            "enhancing_tumor": "<50% decrease to <25% increase",
            "non_enhancing_tumor": "Stable",
            "new_lesions": "None",
            "corticosteroids": "Stable or decreased",
            "clinical_status": "Stable",
        },
    },
    "PD": {
        "name": "Progressive Disease",
        "description": (
            ">=25% increase in the sum of products of perpendicular diameters "
            "of enhancing lesions. Or significant increase in non-enhancing "
            "FLAIR/T2 lesions. Or any new lesions. Or clinical deterioration "
            "not attributable to other causes."
        ),
        "requirements": {
            "enhancing_tumor": ">=25% increase",
            "non_enhancing_tumor": "Significant increase",
            "new_lesions": "Present",
            "corticosteroids": "N/A (determination independent)",
            "clinical_status": "Deteriorated",
        },
    },
}


def evaluate_response(
    tumor_size_change_pct: float = 0.0,
    contrast_enhancement: str = "stable",
    new_lesions: bool = False,
    clinical_condition: str = "stable",
    non_enhancing_change: str = "stable",
    corticosteroid_change: str = "stable",
) -> dict:
    """
    Evaluate tumor treatment response using RANO criteria.

    Args:
        tumor_size_change_pct: Percentage change in tumor size
            (negative = shrinkage, positive = growth). E.g. -60 means 60% shrinkage.
        contrast_enhancement: One of 'absent', 'decreased', 'stable', 'increased'
        new_lesions: Whether new lesions are detected
        clinical_condition: One of 'improved', 'stable', 'deteriorated'
        non_enhancing_change: One of 'decreased', 'stable', 'increased'
        corticosteroid_change: One of 'decreased', 'stable', 'increased', 'none'

    Returns:
        dict with assessment, criteria details, and reasoning
    """
    reasoning = []
    assessment = None

    # --- Progressive Disease (check first, most critical) ---
    if (
        tumor_size_change_pct >= 25
        or new_lesions
        or clinical_condition == "deteriorated"
        or non_enhancing_change == "increased"
    ):
        assessment = "PD"
        if tumor_size_change_pct >= 25:
            reasoning.append(
                f"Tumor size increased by {tumor_size_change_pct:.1f}% (>=25% threshold for PD)")
        if new_lesions:
            reasoning.append("New lesions detected - automatic PD criterion")
        if clinical_condition == "deteriorated":
            reasoning.append("Clinical condition deteriorated")
        if non_enhancing_change == "increased":
            reasoning.append("Non-enhancing tumor/FLAIR signal increased")

    # --- Complete Response ---
    elif (
        tumor_size_change_pct <= -99
        and contrast_enhancement in ("absent", "decreased")
        and not new_lesions
        and clinical_condition in ("improved", "stable")
        and non_enhancing_change in ("decreased", "stable")
    ):
        assessment = "CR"
        reasoning.append("Complete disappearance of enhancing disease")
        reasoning.append("No new lesions, clinical condition stable/improved")

    # --- Partial Response ---
    elif (
        tumor_size_change_pct <= -50
        and not new_lesions
        and clinical_condition in ("improved", "stable")
    ):
        assessment = "PR"
        reasoning.append(
            f"Tumor size decreased by {abs(tumor_size_change_pct):.1f}% (>=50% decrease)")
        reasoning.append("No new lesions, clinical condition stable/improved")

    # --- Stable Disease (default) ---
    else:
        assessment = "SD"
        reasoning.append(
            f"Tumor size change: {tumor_size_change_pct:+.1f}% "
            f"(between -50% and +25%)")
        reasoning.append("Does not meet criteria for CR, PR, or PD")

    # Build result
    criteria_info = RANO_CRITERIA[assessment].copy()
    result = {
        "assessment": assessment,
        "assessment_name": criteria_info["name"],
        "description": criteria_info["description"],
        "reasoning": reasoning,
        "input_summary": {
            "tumor_size_change_pct": tumor_size_change_pct,
            "contrast_enhancement": contrast_enhancement,
            "new_lesions": new_lesions,
            "clinical_condition": clinical_condition,
            "non_enhancing_change": non_enhancing_change,
            "corticosteroid_change": corticosteroid_change,
        },
        "criteria_requirements": criteria_info["requirements"],
    }

    return result


# --- MCP Server (optional standalone mode) ----------------------------

def run_mcp_server():
    """Run as a standalone MCP server via stdio transport."""
    try:
        from mcp.server.fastmcp import FastMCP
    except ImportError:
        print("ERROR: 'mcp' package not installed. pip install mcp[cli]")
        return

    mcp = FastMCP("RANO Criteria Evaluator")

    @mcp.tool()
    def rano_evaluate_response(
        tumor_size_change_pct: float = 0.0,
        contrast_enhancement: str = "stable",
        new_lesions: bool = False,
        clinical_condition: str = "stable",
        non_enhancing_change: str = "stable",
        corticosteroid_change: str = "stable",
    ) -> dict:
        """
        Evaluate tumor treatment response using RANO criteria.
        Provide tumor size change percentage (negative = shrinkage),
        enhancement status, lesion info, and clinical condition.
        Returns assessment (CR/PR/SD/PD) with reasoning.
        """
        return evaluate_response(
            tumor_size_change_pct=tumor_size_change_pct,
            contrast_enhancement=contrast_enhancement,
            new_lesions=new_lesions,
            clinical_condition=clinical_condition,
            non_enhancing_change=non_enhancing_change,
            corticosteroid_change=corticosteroid_change,
        )

    mcp.run(transport="stdio")


if __name__ == "__main__":
    run_mcp_server()


In [ ]:
%%writefile mcp_servers/cap_report.py
"""
CAP Structured Reporting MCP Server
======================================
Implements CAP (College of American Pathologists) structured report
template for brain tumor MRI studies.

Auto-generates all 9 CAP report sections from pipeline state.

Can run as:
  - Standalone MCP server (stdio mode)
  - Direct function call (in-process)
"""
import json
from datetime import datetime


def generate_cap_report(state: dict) -> dict:
    """
    Generate a structured CAP report from the full pipeline state.

    Returns a dict with all 9 CAP sections.
    """
    patient_id = state.get("patient_id", "unknown")
    clinical = state.get("clinical_profile", {})
    analysis = state.get("tumor_analysis", {})
    radiomics = state.get("radiomics_features", {})
    similar = state.get("similar_cases", [])
    reasoning = state.get("clinical_reasoning", "")
    corrections = state.get("physician_corrections", {})
    history = state.get("patient_history", [])

    who = analysis.get("who_classification", {})
    rano = analysis.get("rano_assessment", {})
    progression = analysis.get("progression", {})
    morph = clinical.get("morphology", {})

    # -- Section 1: Patient Information --
    scan_history_count = len(history)
    section_patient = {
        "patient_id": patient_id,
        "prior_scans": scan_history_count,
        "report_date": datetime.now().strftime("%Y-%m-%d"),
        "report_time": datetime.now().strftime("%H:%M:%S"),
        "institution": "Agentic Clinical Brain Tumor Intelligence Platform",
        "pipeline_version": "3.0",
    }

    # -- Section 2: MRI Study Information --
    section_study = {
        "modalities": ["T1", "T1CE", "T2", "FLAIR"],
        "segmentation_method": "DynUNet (MONAI) / Ground-truth fallback",
        "preprocessing": [
            "N4 Bias Field Correction",
            "Skull Stripping",
            "Z-score Intensity Normalization",
            "1mm Isotropic Resampling",
        ],
    }

    # -- Section 3: Tumor Characteristics --
    section_tumor = {
        "location": clinical.get("tumor_location", []),
        "primary_location": clinical.get("primary_location", "unknown"),
        "volume_mm3": morph.get("tumor_volume", 0),
        "volume_severity": clinical.get("volume_severity", "unknown"),
        "max_diameter_mm": morph.get("max_diameter", 0),
        "sphericity": morph.get("sphericity", 0),
        "surface_area_mm2": morph.get("surface_area", 0),
    }

    # -- Section 4: Radiomics Summary --
    rad_summary = clinical.get("radiomics_summary", {})
    section_radiomics = {
        "total_features_extracted": rad_summary.get("num_features", len(radiomics)),
        "shape_features": rad_summary.get("key_shape", {}),
        "intensity_features": rad_summary.get("key_intensity", {}),
        "texture_features_count": len(rad_summary.get("key_texture", {})),
    }

    # -- Section 5: RANO Classification --
    section_rano = {
        "assessment": rano.get("assessment", "N/A"),
        "assessment_name": rano.get("assessment_name", "Not assessed"),
        "reasoning": rano.get("reasoning", []),
        "physician_override": rano.get("physician_override", False),
        "progression_state": progression.get("progression_state", "unknown"),
        "growth_rate_mm3_per_day": progression.get("growth_rate"),
        "size_change_pct": progression.get("size_change_pct"),
    }

    # -- Section 6: WHO Classification --
    section_who = {
        "classified_as": who.get("classified_as", "unknown"),
        "who_grade": who.get("who_grade", "unknown"),
        "full_name": who.get("full_name", ""),
        "confidence": who.get("confidence", 0),
        "description": who.get("description", ""),
        "prognosis": who.get("prognosis", ""),
        "standard_treatment": who.get("standard_treatment", ""),
        "differential_diagnosis": who.get("differential", []),
        "physician_override": who.get("physician_override", False),
    }

    # -- Section 7: Similar Tumor Cases --
    section_similar = {
        "retrieval_method": (
            "Weaviate vector similarity" if similar else "NumPy cosine fallback"
        ),
        "cases_retrieved": len(similar),
        "top_cases": [
            {
                "patient_id": c.get("patient_id", ""),
                "location": c.get("tumor_location", []),
                "severity": c.get("volume_severity", ""),
                "similarity_score": c.get("similarity", c.get("distance")),
            }
            for c in similar[:5]
        ],
    }

    # -- Section 8: Clinical Interpretation --
    section_interpretation = {
        "ai_clinical_reasoning": reasoning,
        "inferred_symptoms": clinical.get("inferred_symptoms", []),
        "reasoning_engine": (
            "Llama 3 (Ollama)" if "DIAGNOSIS ASSESSMENT" not in reasoning else "Rule-based fallback"
        ),
    }

    # -- Section 9: Physician Notes --
    section_physician = {
        "review_status": corrections.get("action", "not_reviewed"),
        "physician": corrections.get("physician", "N/A"),
        "notes": corrections.get("physician_notes", ""),
        "segmentation_edited": corrections.get("segmentation_edited", False),
        "review_timestamp": corrections.get("timestamp", ""),
    }

    # -- Assemble Full Report --
    cap_report = {
        "cap_report_type": "CAP Structured MRI Brain Tumor Report",
        "section_1_patient_information": section_patient,
        "section_2_mri_study_information": section_study,
        "section_3_tumor_characteristics": section_tumor,
        "section_4_radiomics_summary": section_radiomics,
        "section_5_rano_classification": section_rano,
        "section_6_who_classification": section_who,
        "section_7_similar_tumor_cases": section_similar,
        "section_8_clinical_interpretation": section_interpretation,
        "section_9_physician_notes": section_physician,
    }

    return cap_report


def run_cap_reporting(state: dict) -> dict:
    """
    LangGraph node: Generate and save the CAP structured report.
    """
    import os
    patient_id = state["patient_id"]
    output_dir = state["output_dir"]
    errors = list(state.get("errors", []))

    cap_dir = os.path.join(output_dir, "reports", "cap")
    os.makedirs(cap_dir, exist_ok=True)

    print(f"[CAP Report] Generating for patient: {patient_id}")

    try:
        cap_report = generate_cap_report(state)
        save_path = os.path.join(cap_dir, f"{patient_id}_cap_report.json")
        with open(save_path, "w") as f:
            json.dump(cap_report, f, indent=2, default=str)
        print(f"  CAP report saved: {save_path}")
        print(f"  WHO: {cap_report['section_6_who_classification']['classified_as']}")
        print(f"  RANO: {cap_report['section_5_rano_classification']['assessment']}")
        return {**state, "cap_report": cap_report, "cap_report_path": save_path, "errors": errors}
    except Exception as e:
        msg = f"CAP report generation failed for {patient_id}: {e}"
        print(f"  [ERROR] {msg}")
        errors.append(msg)
        return {**state, "cap_report": {}, "errors": errors}


# --- MCP Server mode --------------------------------------------------

def run_mcp_server():
    """Run as standalone MCP server."""
    try:
        from mcp.server.fastmcp import FastMCP
    except ImportError:
        print("ERROR: 'mcp' package not installed. pip install mcp[cli]")
        return

    mcp = FastMCP("CAP Brain Tumor Report Generator")

    @mcp.tool()
    def generate_cap_structured_report(state_json: str) -> str:
        """
        Generate a full CAP structured report from a patient pipeline state JSON string.
        Returns the report as a JSON string with all 9 CAP sections.
        """
        state = json.loads(state_json)
        report = generate_cap_report(state)
        return json.dumps(report, indent=2, default=str)

    mcp.run(transport="stdio")


if __name__ == "__main__":
    run_mcp_server()


In [ ]:
%%writefile mcp_servers/__init__.py
# empty init


## 2. Load Evaluation Data
Upload **both** data sources to Colab for best results:

1. **Segmentation metrics** (better quality): `brats_outputs/evaluation_results.csv`
2. **Radiomics features** (shape/texture): 4x `evaluation_results_batch_*.csv`

The notebook **merges** them: radiomics features for WHO classification + better segmentation scores for evaluation.

In [ ]:
import pandas as pd
import os
import json
import glob
import numpy as np
from mcp_servers.who_classification import classify_tumor
from mcp_servers.rano_criteria import evaluate_response
from mcp_servers.cap_report import generate_cap_report

# === Load BOTH data sources and merge for best results ===

# 1. Load segmentation metrics from brats_outputs (BETTER segmentation quality)
seg_candidates = [
    '/content/brats_outputs/evaluation_results.csv',
    '/content/evaluation_results.csv',
]
seg_df = None
for p in seg_candidates:
    if os.path.exists(p):
        seg_df = pd.read_csv(p)
        print(f'Segmentation: Loaded {len(seg_df)} patients from {os.path.basename(p)}')
        break

# 2. Load radiomics batch CSVs (for shape/texture features)
batch_files = sorted(glob.glob('/content/evaluation_results_batch_*.csv'))
rad_df = None
if batch_files:
    print(f'Radiomics: Found {len(batch_files)} batch CSVs')
    rad_df = pd.concat([pd.read_csv(f) for f in batch_files], ignore_index=True)
    print(f'Radiomics: Merged {len(rad_df)} patients ({len(rad_df.columns)} columns)')

# 3. Merge: radiomics features + BETTER segmentation metrics from brats_outputs
if rad_df is not None and seg_df is not None:
    rad_df['_merge_id'] = rad_df['subject'].astype(str)
    seg_df['_merge_id'] = seg_df['patient_id'].astype(str)
    seg_cols_to_replace = ['dice', 'hd95', 'sensitivity', 'specificity']
    for col in seg_cols_to_replace:
        if col in rad_df.columns:
            rad_df = rad_df.drop(columns=[col])
    df = rad_df.merge(
        seg_df[['_merge_id'] + seg_cols_to_replace + ['pred_voxels', 'gt_voxels', 'tumor_frac']],
        on='_merge_id', how='left'
    )
    df = df.drop(columns=['_merge_id'])
    patient_id_col = 'subject'
    data_source = 'merged'
    print(f'MERGED dataset: {len(df)} patients')
    print('  Radiomics features: from batch CSVs')
    print('  Segmentation metrics: from brats_outputs (better quality)')
elif rad_df is not None:
    df = rad_df
    patient_id_col = 'subject'
    data_source = 'radiomics'
    print(f'Using radiomics-only data ({len(df)} patients)')
elif seg_df is not None:
    df = seg_df
    patient_id_col = 'patient_id'
    data_source = 'segmentation'
    print(f'Using segmentation-only data ({len(df)} patients)')
else:
    print('ERROR: No data found!')
    df = pd.DataFrame()
    patient_id_col = 'subject'
    data_source = None

if not df.empty:
    print(f'Data source mode: {data_source}')
    print(f'Patient ID column: {patient_id_col}')
    print(f'Total columns: {len(df.columns)}')

## 3. Run the Clinical Pipeline\nMaps PyRadiomics features to the WHO classifier, runs RANO criteria, and generates CAP reports.

In [ ]:
import os

mcp_results = []
cap_reports_generated = 0

os.makedirs('/content/reports/cap', exist_ok=True)

print(f'Starting MCP Pipeline on {len(df)} patients (source: {data_source})...')
print('=' * 60)

for index, row in df.iterrows():
    subject_id = str(row.get(patient_id_col, f'Patient_{index}'))

    # Map features based on data source
    if data_source in ('radiomics', 'merged'):
        morphology = {
            'tumor_volume': row.get('pred_original_shape_MeshVolume', 0),
            'sphericity': row.get('pred_original_shape_Sphericity', 0.5),
            'max_diameter': row.get('pred_original_shape_Maximum3DDiameter', 0),
            'surface_area': row.get('pred_original_shape_SurfaceArea', 0)
        }
        variance = row.get('pred_original_firstorder_Variance', 0)
        radiomics_patterns = {
            'intensity_std': np.sqrt(variance) if variance > 0 else 0,
            'intensity_skewness': row.get('pred_original_firstorder_Skewness', 0)
        }
        num_extracted = len([c for c in df.columns if c.startswith('pred_original_')])
    else:
        vol = row.get('pred_voxels', 0)
        morphology = {
            'tumor_volume': vol,
            'sphericity': 0.5,
            'max_diameter': (vol * 3 / (4 * np.pi)) ** (1/3) * 2 if vol > 0 else 0,
            'surface_area': 0
        }
        radiomics_patterns = {'intensity_std': 0, 'intensity_skewness': 0}
        num_extracted = 0

    molecular = {
        'IDH': 'unknown', 'MGMT': 'unknown', '1p19q': 'unknown',
        'ATRX': 'unknown', 'TP53': 'unknown'
    }

    who_result = classify_tumor(morphology, radiomics_patterns, None, molecular)

    rano_result = evaluate_response(
        tumor_size_change_pct=0.0,
        contrast_enhancement='stable',
        new_lesions=False,
        clinical_condition='stable'
    )

    dice_val = row.get('dice', None)
    hd95_val = row.get('hd95', None)
    sens_val = row.get('sensitivity', None)
    spec_val = row.get('specificity', None)

    vol_severity = 'small'
    if morphology['tumor_volume'] > 50000:
        vol_severity = 'large'
    elif morphology['tumor_volume'] > 10000:
        vol_severity = 'moderate'

    state = {
        'patient_id': subject_id,
        'clinical_profile': {
            'morphology': morphology,
            'tumor_location': ['unknown'],
            'primary_location': 'brain',
            'volume_severity': vol_severity
        },
        'tumor_analysis': {
            'who_classification': who_result,
            'rano_assessment': rano_result,
            'progression': {'progression_state': 'stable'}
        },
        'radiomics_features': {'num_extracted': num_extracted},
        'segmentation_metrics': {
            'dice': dice_val, 'hd95': hd95_val,
            'sensitivity': sens_val, 'specificity': spec_val,
        },
    }

    cap_report = generate_cap_report(state)

    report_path = f'/content/reports/cap/{subject_id}_cap_report.json'
    with open(report_path, 'w') as f:
        json.dump(cap_report, f, indent=2)
    cap_reports_generated += 1

    mcp_results.append({
        'subject': subject_id,
        'who_diagnosis': who_result.get('classified_as', 'unknown'),
        'who_grade': who_result.get('who_grade', 'unknown'),
        'who_confidence': who_result.get('confidence', 0),
        'who_full_name': who_result.get('full_name', ''),
        'rano_assessment': rano_result.get('assessment', 'SD'),
        'tumor_volume': morphology['tumor_volume'],
        'sphericity': morphology['sphericity'],
        'dice': dice_val,
        'hd95': hd95_val,
        'cap_report_file': report_path
    })

    if (index + 1) % 50 == 0:
        print(f'  Processed {index + 1}/{len(df)} patients...')

print('=' * 60)
print(f'DONE: Generated {cap_reports_generated} CAP Structured Reports.')
mcp_df = pd.DataFrame(mcp_results)
mcp_df.to_csv('/content/mcp_clinical_summary.csv', index=False)
print('Summary saved to /content/mcp_clinical_summary.csv')

## 4. Evaluation Metrics\nView the distribution of tumor classifications and AI confidence scores.

In [ ]:
if not mcp_df.empty:
    print("=========================================")
    print("      WHO CLASSIFICATION RESULTS         ")
    print("=========================================")
    print(mcp_df['who_diagnosis'].value_counts())
    
    print("\n=========================================")
    print("      AVERAGE AI CONFIDENCE              ")
    print("=========================================")
    print(f"{mcp_df['who_confidence'].mean() * 100:.2f}%")
    
    print("\n=========================================")
    print("      RANO ASSESSMENTS                   ")
    print("=========================================")
    print(mcp_df['rano_assessment'].value_counts())

## 5. Export CAP Reports as ZIP

In [ ]:
import shutil
from google.colab import files

# Zip all the JSON reports
shutil.make_archive('/content/cap_reports', 'zip', '/content/reports/cap')
print('Zipped all CAP reports to /content/cap_reports.zip')

# Download
files.download('/content/cap_reports.zip')
files.download('/content/mcp_clinical_summary.csv')